# Data Integration and Quality Assessment — Owner B

This notebook implements the reproducible cleaning work for the TfNSW EV charger
dataset. It reads the acquisition output from `data/raw/`, preserves raw values,
and writes auditable cleaned intermediates to `data/interim/`.

Spatial joining is intentionally left to `src/integration/spatial_join.py` / the
spatial owner, after this notebook's cleaned output is available.


## Running this notebook

Run from the repository root after the acquisition stage has created:

- `data/raw/ev_charging_locations.csv`
- `data/raw/sa4_shapefile/` (used later by the spatial-join stage)

This notebook does not alter `data/raw/`. It produces `data/interim/` files that
are regenerated rather than committed to Git.


In [ ]:
from pathlib import Path
import hashlib
import json

import pandas as pd

try:
    from IPython.display import display
except ImportError:
    def display(value):
        print(value)

# This cell assumes the notebook is run from the repository root.
PROJECT_ROOT = Path.cwd()
RAW_CSV = PROJECT_ROOT / "data" / "raw" / "ev_charging_locations.csv"
INTERIM_DIR = PROJECT_ROOT / "data" / "interim"
INTERIM_DIR.mkdir(parents=True, exist_ok=True)

CLEANED_CSV = INTERIM_DIR / "ev_chargers_cleaned.csv"
COMPONENTS_CSV = INTERIM_DIR / "charger_power_components.csv"
QUALITY_JSON = INTERIM_DIR / "cleaning_quality_summary.json"

raw = pd.read_csv(RAW_CSV, dtype=str, keep_default_na=False)
assert len(raw) == 1958, "Expected the December 2025 source release with 1,958 rows."
raw.head()


## 1. Raw-data quality profile

Record the source issues before cleaning. These counts are used directly in the
report's Data Cleaning and Quality Assessment section.


In [ ]:
missing_before = raw.replace(r"^\s*$", pd.NA, regex=True).isna().sum()
operator_counts_before = raw["Operator"].value_counts()
rating_counts_before = raw["Charger_rating"].value_counts()
duplicate_key_columns = ["Station_name", "Latitude", "Longitude"]
duplicate_groups_before = (
    raw.groupby(duplicate_key_columns, dropna=False)
    .size()
    .reset_index(name="group_size")
    .query("group_size > 1")
    .sort_values(duplicate_key_columns)
)

display(missing_before.rename("missing_values").to_frame())
display(raw[["Operator", "Charger_Type", "Charger_rating"]].nunique().rename("unique_values").to_frame())
display(raw["Operator"].str.len().value_counts().sort_index().rename("operator_length_rows").to_frame())
display(duplicate_groups_before)


## 2. Operator canonicalization and stable record IDs

`Operator_raw` retains source text. Only approved evidence-based merges are applied:
exact-prefix relationships, whitespace/case variants, and `Charge Hub` spacing.
Unrecoverable truncations remain raw, while `University of` becomes `Unknown/missing`
because it is venue text from the address rather than an operator.


In [ ]:
RAW_TO_CANONICAL = {
    "360 EV Charge": "360 EV Charge", "AXCharge": "AXCharge", "Alchemy Charge": "Alchemy Charge",
    "Ampol": "Ampol", "BMW": "BMW", "BP": "BP Australia", "BP Australia ": "BP Australia",
    "CasaCharge": "CasaCharge", "Charge Hub": "ChargeHub", "Charge OS": "Charge OS",
    "ChargeHub": "ChargeHub", "ChargePoint": "ChargePoint", "ChargePost": "ChargePost",
    "Chargefox": "Chargefox", "Chargestar": "Chargestar", "Counties Energy": "Counties Energy",
    "EVE Australia": "EVE Australia", "EV Meter": "EV Meter", "EVNet": "EVNet", "EVSE": "EVSE",
    "EVUp": "EVUp", "EVX": "EVX", "Elanga": "Elanga", "Energy Austra": "Energy Austra",
    "Engie": "Engie", "Everty": "Everty", "Evie": "Evie Networks", "Evie Networks": "Evie Networks",
    "Exploren": "Exploren", "Fast Cities A": "Fast Cities A", "Gentari": "Gentari", "JOLT": "JOLT",
    "NRMA": "NRMA Electric", "NRMA Electric": "NRMA Electric", "Non-Networked": "Non-networked",
    "Non-networked": "Non-networked", "Noodoe": "Noodoe", "PLUS ES": "PLUS ES",
    "PLUS ES Manag": "PLUS ES Manag", "Porsche Destination Charging": "Porsche Destination Charging",
    "Porsche Smart Mobility": "Porsche Smart Mobility", "Saascharge": "Saascharge",
    "Smart Charge": "Smart Charge", "Tesla": "Tesla Motors", "Tesla Motors ": "Tesla Motors",
    "University of": "Unknown/missing", "Viva Energy A": "Viva Energy Australia",
    "Viva Energy Australia": "Viva Energy Australia", "Wevolt": "Wevolt", "Zeus Renewables": "Zeus Renewables",
}

cleaned = raw.copy()
assert set(cleaned["Operator"]) == set(RAW_TO_CANONICAL), "Review the mapping if the acquisition source changes."
cleaned.insert(cleaned.columns.get_loc("Operator") + 1, "Operator_raw", cleaned["Operator"])
cleaned["Operator"] = cleaned["Operator_raw"].map(RAW_TO_CANONICAL)
assert cleaned["Operator"].notna().all()

ID_FIELDS = [
    "OBJECTID", "Station_name", "Station_address", "Operator_raw", "Number_of_plugs",
    "Charger_Type", "Charger_rating", "Latitude", "Longitude", "LGANAME", "PCODE", "Source",
]

def make_charger_record_id(row: pd.Series) -> str:
    payload = ["charger_record_id_v1", [(field, row[field]) for field in ID_FIELDS]]
    return "chr_" + hashlib.sha256(
        json.dumps(payload, ensure_ascii=False, separators=(",", ":")).encode("utf-8")
    ).hexdigest()

cleaned.insert(0, "charger_record_id", cleaned.apply(make_charger_record_id, axis=1))
assert cleaned["charger_record_id"].is_unique
print(f"Operators: {raw['Operator'].nunique()} raw -> {cleaned['Operator'].nunique()} canonical")


## 3. Charger-rating normalization

Preserve `Charger_rating_raw`; expose normalized scalar power only where justified.
Multi-plug configurations are written to a separate child table, preserving the
one-to-many relationship between a charger and its rated plug groups.


In [ ]:
PROPER_KW = r"^(\d+) kW$"
UNITLESS_KW = r"^(\d+)$"
MULTI_PLUG = r"^(\d+)x(\d+)kW\s*&\s*(\d+)x(\d+)kW$"

cleaned.insert(cleaned.columns.get_loc("Charger_rating") + 1, "Charger_rating_raw", cleaned["Charger_rating"])
raw_rating = cleaned["Charger_rating_raw"]
proper = raw_rating.str.fullmatch(PROPER_KW)
unitless = raw_rating.str.fullmatch(UNITLESS_KW)
ac_placeholder = raw_rating.eq("AC")
multi_parts = raw_rating.str.extract(MULTI_PLUG)
multi = multi_parts[0].notna()
assert (proper.astype(int) + unitless.astype(int) + ac_placeholder.astype(int) + multi.astype(int)).eq(1).all()

normalized = pd.Series(pd.NA, index=cleaned.index, dtype="string")
rating_kw = pd.Series(pd.NA, index=cleaned.index, dtype="Int64")
genuinely_unknown = pd.Series(False, index=cleaned.index, dtype="boolean")
missing_reason = pd.Series("not_missing", index=cleaned.index, dtype="string")
representation = pd.Series(pd.NA, index=cleaned.index, dtype="string")
implied_plugs = pd.Series(pd.NA, index=cleaned.index, dtype="Int64")
reported_minus_implied = pd.Series(pd.NA, index=cleaned.index, dtype="Int64")
scope_ambiguous = pd.Series(pd.NA, index=cleaned.index, dtype="boolean")

for mask, pattern in [(proper, PROPER_KW), (unitless, UNITLESS_KW)]:
    values = raw_rating.loc[mask].str.extract(pattern)[0].astype("Int64")
    normalized.loc[mask] = values.astype("string") + " kW"
    rating_kw.loc[mask] = values
    representation.loc[mask] = "single_power_kw"

representation.loc[ac_placeholder] = "unknown"
genuinely_unknown.loc[ac_placeholder] = True
missing_reason.loc[ac_placeholder] = "ac_restates_charger_type"

implied_plugs.loc[multi] = multi_parts.loc[multi, 0].astype("Int64") + multi_parts.loc[multi, 2].astype("Int64")
reported = pd.to_numeric(cleaned.loc[multi, "Number_of_plugs"], errors="raise").astype("Int64")
reported_minus_implied.loc[multi] = reported - implied_plugs.loc[multi]
scope_ambiguous.loc[multi] = reported_minus_implied.loc[multi].gt(0)
representation.loc[multi] = "multi_plug_configuration"

cleaned["charger_rating_normalized"] = normalized
cleaned["charger_rating_kw"] = rating_kw
cleaned["rating_genuinely_unknown"] = genuinely_unknown
cleaned["rating_missing_reason"] = missing_reason
cleaned["rating_representation"] = representation
cleaned["rating_implied_plug_count"] = implied_plugs
cleaned["reported_minus_implied_plugs"] = reported_minus_implied
cleaned["plug_count_scope_ambiguous"] = scope_ambiguous

components = []
for index in cleaned.index[multi]:
    values = multi_parts.loc[index]
    for component_index, count_col, power_col in [(1, 0, 1), (2, 2, 3)]:
        components.append({
            "charger_record_id": cleaned.at[index, "charger_record_id"],
            "component_index": component_index,
            "plug_count": int(values[count_col]),
            "power_kw": int(values[power_col]),
        })
charger_power_components = pd.DataFrame(components)
assert len(charger_power_components) == 198
assert not charger_power_components.duplicated(["charger_record_id", "component_index"]).any()
display(cleaned["rating_representation"].value_counts().rename("row_count").to_frame())


## 4. Duplicate classification

The following is an approved, auditable status classification. It does not delete
any source row. Spatial integration should consume the non-superseded records, while
the manual-review records remain visible for human validation.


In [ ]:
SUPERSEDED_BY = {
    # Figtree: retain the more detailed multi-plug configuration.
    "chr_cc8db9e86339847395b0df4732f7d27b8b3b033b7f4b65a36fb283fb213c3bb3": "chr_af9c025da56f4cc2d561714de7246dc6bb23686e6b512e6ab05d199a2f57f690",
    # Kempsey: retain known 22 kW rather than the AC placeholder.
    "chr_6f03ded1946613628b5ef5c8e518915afa764f01fad981eac5ca5b5fe77542a0": "chr_b070320b6df294a808c6d40cb378730b74652d2beff3cdf31125d399490e4a40",
}
MANUAL_REVIEW_IDS = {
    # Albury City Council, Hay Shire Council, and University of Wollongong.
    "chr_754742a5efb69a89e35202973e172f6a764320d26fc5e47445efadeca031cebe",
    "chr_10e5473a508a28647a75d953e74487c6c4d9b6cdd4af9348e9e1f8c4d4df9457",
    "chr_ef926dbadf6ffbe694eac1581b8db7736dca9dc48d956df6cec41f91fbb1d45a",
    "chr_704a97b0bb05d6c2390ba490cff23347c71eb31df86782a464a32069a221752e",
    "chr_0e54c9e24c9f9ffd735a39ccad36aa314503727e8dbd2aa4ee87ed26bdd12a12",
    "chr_474bbfa745261fafd0e8a26c2dcd3f315b1b846e270c4ed7f8d9939f7f7b9784",
}

cleaned["duplicate_status"] = "active"
cleaned["superseded_by"] = ""
superseded = cleaned["charger_record_id"].isin(SUPERSEDED_BY)
cleaned.loc[superseded, "duplicate_status"] = "superseded"
cleaned.loc[superseded, "superseded_by"] = cleaned.loc[superseded, "charger_record_id"].map(SUPERSEDED_BY)
cleaned.loc[cleaned["charger_record_id"].isin(MANUAL_REVIEW_IDS), "duplicate_status"] = "flagged_manual_review"

assert len(cleaned) == len(raw)
assert cleaned.loc[cleaned["duplicate_status"].ne("superseded"), "superseded_by"].eq("").all()
display(cleaned["duplicate_status"].value_counts().rename("row_count").to_frame())


## 5. Write auditable integration outputs

These outputs become the input to the spatial-join module. `quality_summary.json`
captures report-ready counts without manually copying results from notebook cells.


In [ ]:
quality_summary = {
    "input_rows": len(raw),
    "output_rows": len(cleaned),
    "unique_charger_record_ids": int(cleaned["charger_record_id"].nunique()),
    "operator_unique_before": int(raw["Operator"].nunique()),
    "operator_unique_after": int(cleaned["Operator"].nunique()),
    "missing_before": missing_before.astype(int).to_dict(),
    "rating_representation": cleaned["rating_representation"].value_counts().to_dict(),
    "duplicate_status": cleaned["duplicate_status"].value_counts().to_dict(),
    "multi_plug_component_rows": len(charger_power_components),
}

cleaned.to_csv(CLEANED_CSV, index=False, encoding="utf-8", lineterminator="\n")
charger_power_components.to_csv(COMPONENTS_CSV, index=False, encoding="utf-8", lineterminator="\n")
QUALITY_JSON.write_text(json.dumps(quality_summary, indent=2), encoding="utf-8")

assert len(pd.read_csv(CLEANED_CSV, dtype=str, keep_default_na=False, engine="python")) == 1958
print("Wrote:")
for path in [CLEANED_CSV, COMPONENTS_CSV, QUALITY_JSON]:
    print("-", path)


## Next hand-off

Commit this notebook and the reusable functions moved into `src/integration/cleaning.py`
and `src/integration/quality.py`. Do not commit the regenerated raw/interim data files;
the repository `.gitignore` already enforces that policy. The spatial owner can then
consume `data/interim/ev_chargers_cleaned.csv` for the SA4 join.
